# 数据分析结论（V3.0）

## 1. 版本演进验证
- V1.0 → V2.0：函数化拆分 + 宏定义消除魔法数字 + 日志系统
- V2.0 → V3.0：多模块拆分(6头文件+6源文件) + Stats结构体 + 双排序算法 + scanf安全加固

## 2. 数据分析发现
- 通过柱状图可以直观看到每位学生分数高低，排序和C语言程序输出排名一致
- 饼图展示成绩等级分布，便于识别优秀/不及格比例
- **交叉校验**：Python计算结果与C程序输出完全一致，验证了C代码的计算正确性

## 3. 代码质量提升
- 模块化架构使代码可复用、可测试、可扩展
- swap_students / validate_score 消除重复代码
- 自动化测试覆盖7个边界用例，保证重构后功能完整

## 4. 后续改进方向
- 支持从CSV文件批量导入学生数据
- 增加更多排序算法（快速排序、归并排序）
- 添加成绩等级评定功能（A/B/C/D/F）

In [ ]:
# 7.交叉校验：Python 统计 vs C程序输出
# 从C程序输出中提取统计值
avg_match = re.search(r"平均分:\s*([\d.]+)", content)
max_match = re.search(r"最高分:\s*(\d+)", content)
min_match = re.search(r"最低分:\s*(\d+)", content)

c_avg = float(avg_match.group(1)) if avg_match else None
c_max = int(max_match.group(1)) if max_match else None
c_min = int(min_match.group(1)) if min_match else None

print("====交叉校验: Python vs C程序====")
print(f"{'指标':<8} {'Python':<10} {'C程序':<10} {'一致':<6}")
print("-" * 38)
for label, py_val, c_val in [
    ("平均分", stats_dict["平均分"], c_avg),
    ("最高分", stats_dict["最高分"], c_max),
    ("最低分", stats_dict["最低分"], c_min),
    ("总分",   stats_dict["总分"],   None),
]:
    match = "✅" if (py_val == c_val or c_val is None) else "❌"
    c_str = f"{c_val}" if c_val is not None else "N/A"
    print(f"{label:<8} {py_val:<10} {c_str:<10} {match:<6}")

# 8.绘制柱状图：Python vs C 统计对比
metrics = ["平均分", "最高分", "最低分"]
py_vals = [stats_dict["平均分"], stats_dict["最高分"], stats_dict["最低分"]]
c_vals = [c_avg, c_max, c_min]

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(metrics))
width = 0.35
bars1 = ax.bar([i - width/2 for i in x], py_vals, width, label="Python计算", color="#4285F4")
bars2 = ax.bar([i + width/2 for i in x], c_vals, width, label="C程序输出", color="#34A853")
ax.set_ylabel("分数")
ax.set_title("Python vs C程序 交叉校验")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
plt.savefig("cross_validation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 5.成绩分段统计：优秀(>=90) / 良好(75-89) / 及格(60-74) / 不及格(<60)
bins = [0, 60, 75, 90, 101]
labels = ["不及格(<60)", "及格(60-74)", "良好(75-89)", "优秀(>=90)"]
df["等级"] = pd.cut(df["成绩"], bins=bins, labels=labels, right=False)
level_counts = df["等级"].value_counts().reindex(labels, fill_value=0)
print("\n====成绩分段统计====")
for level, count in level_counts.items():
    print(f"{level}: {count}人 ({count/len(df)*100:.1f}%)")

# 6.绘制饼图：成绩等级分布
plt.figure(figsize=(8, 8))
colors_pie = ["#EA4335", "#FBBC04", "#34A853", "#4285F4"]
plt.pie(level_counts[level_counts > 0], labels=level_counts[level_counts > 0].index,
        autopct="%1.1f%%", colors=colors_pie[:sum(level_counts > 0)],
        startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
plt.title("学生成绩等级分布")
plt.savefig("score_pie.png", dpi=300, bbox_inches="tight")
plt.show()

# 学生成绩数据分析 — V3.0
# 读取C程序输出日志，数据解析、统计指标、可视化绘图
# 支持交叉校验：Python计算结果 vs C程序输出